IMPORT LIBRARY

In [2]:
import tensorflow as tf

from tensorflow.keras.layers import Dense, Dropout, Input, Layer
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import Callback
from tensorflow.keras.utils import register_keras_serializable
from tensorflow.keras.regularizers import l2

CUSTOM LAYER

In [3]:
@register_keras_serializable()
class IngredientsImportanceLayer(Layer):

    def __init__(self, factor=1.2, **kwargs):
        super().__init__(**kwargs)
        self.factor = factor

    def call(self, inputs):
        return inputs * self.factor

    def get_config(self):
        config = super().get_config()
        config.update({
            "factor": self.factor
        })
        return config

CUSTOM LOSS FUNCTION

In [4]:
@register_keras_serializable()
def custom_recipe_loss(y_true, y_pred):
    loss = tf.keras.losses.sparse_categorical_crossentropy(
        y_true,
        y_pred
    )
    return tf.reduce_mean(loss)

CUSTOM CALLBACK

In [5]:
class TrainingLogger(Callback):

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}

        print(
            f"\nEpoch {epoch + 1} selesai | "
            f"Loss: {logs.get('loss', 0):.4f} | "
            f"Accuracy: {logs.get('accuracy', 0):.4f}"
        )

BUILD MODEL MLP

In [6]:
def build_mlp_model(input_dim=5000, num_classes=8):
    input_layer = Input(shape=(input_dim,))

    x = IngredientsImportanceLayer()(input_layer)

    x = Dense(
        128,
        activation="relu",
        kernel_regularizer=l2(0.001)
    )(x)

    x = Dropout(0.4)(x)

    x = Dense(
        64,
        activation="relu",
        kernel_regularizer=l2(0.001)
    )(x)

    x = Dropout(0.4)(x)

    output_layer = Dense(
        num_classes,
        activation="softmax"
    )(x)

    model = Model(
        inputs=input_layer,
        outputs=output_layer
    )

    model.compile(
        optimizer="adam",
        loss=custom_recipe_loss,
        metrics=["accuracy"]
    )

    return model

In [7]:
model = build_mlp_model(
    input_dim=1685,
    num_classes=8
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1685)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ingredients_importance_layer    │ (None, 1685)           │             0 │
│ (IngredientsImportanceLayer)    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       215,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │           520 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 224,584 (877.28 KB)

 Trainable params: 224,584 (877.28 KB)

 Non-trainable params: 0 (0.00 B)